# Feature Selection and Classification Add-on

本 notebook 片段用于接在多受试者特征提取之后。

核心原则：

1. **Subject-wise validation（按受试者验证）**：同一个受试者的所有窗口不能同时出现在训练集和测试集。
2. **Feature selection inside CV（交叉验证内部特征筛选）**：每个fold只用训练集筛选特征，避免信息泄漏。
3. **Small-sample models（小样本模型）**：优先使用逻辑回归、SVM、随机森林等传统模型，不建议深度学习。

In [ ]:
# ============================================================
# 0. Import classification module
# ============================================================

import pandas as pd
import numpy as np

from ppvmd_feature_classification import (
    aggregate_to_subject_level,
    evaluate_subject_level_classification,
    evaluate_window_level_subjectwise_classification,
    run_full_feature_selection_classification_workflow,
)

## 1. Prepare multi-subject feature table

你需要先把所有受试者的特征合并成一个表。

如果你已经有 `all_window_features.csv`，直接读取。

必要列：

- `SubjectID`：受试者编号
- `Label`：0=健康，1=患者/STC
- `WindowMethod`：event_guided 或 fixed
- 其他列：特征

In [ ]:
# ============================================================
# 1. Load or concatenate multi-subject features
# ============================================================

# 示例1：如果你已经有合并好的窗口级特征表
# all_window_features = pd.read_csv("all_subject_window_features.csv")

# 示例2：如果你有多个CSV，可以这样合并
# import glob
# files = glob.glob("*_event_fixed_features.csv")
# all_window_features = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# 请替换为你自己的变量
# all_window_features.head()

## 2. Subject-level aggregation（受试者级聚合）

如果输入是窗口级特征，先聚合为受试者级特征。

这是避免 **pseudo-replication（伪重复）** 的关键步骤。

In [ ]:
# ============================================================
# 2. Aggregate window-level features to subject-level features
# ============================================================

subject_feature_df = aggregate_to_subject_level(
    all_window_features,
    subject_col="SubjectID",
    label_col="Label",
    method_col="WindowMethod",
    aggregations=("mean", "std", "median"),
)

print(subject_feature_df.shape)
subject_feature_df.head()

## 3. Subject-level classification（推荐主结果）

这是论文中最应该报告的结果：每个受试者一行，使用 Leave-One-Out（留一受试者法）交叉验证。

In [ ]:
# ============================================================
# 3. Subject-level classification: event-guided features
# ============================================================

result_event_subject = evaluate_subject_level_classification(
    subject_feature_df,
    label_col="Label",
    subject_col="SubjectID",
    method_col="WindowMethod",
    target_method="event_guided",
    top_k=15,
    corr_threshold=0.90,
    max_missing_rate=0.30,
)

print("Classification summary: event-guided subject-level")
display(result_event_subject.summary)

print("Selected feature frequency")
display(result_event_subject.selected_feature_frequency.head(30))

In [ ]:
# ============================================================
# 4. Subject-level classification: fixed-window baseline
# ============================================================

result_fixed_subject = evaluate_subject_level_classification(
    subject_feature_df,
    label_col="Label",
    subject_col="SubjectID",
    method_col="WindowMethod",
    target_method="fixed",
    top_k=15,
    corr_threshold=0.90,
    max_missing_rate=0.30,
)

print("Classification summary: fixed subject-level")
display(result_fixed_subject.summary)

print("Selected feature frequency")
display(result_fixed_subject.selected_feature_frequency.head(30))

## 5. Window-level subject-wise validation（可作为补充）

窗口可以作为样本，但验证必须按受试者分组。

也就是说，同一个人的所有窗口必须同时在训练集或测试集，不能随机拆开。

In [ ]:
# ============================================================
# 5. Window-level classification with Leave-One-Subject-Out groups
# ============================================================

result_event_window = evaluate_window_level_subjectwise_classification(
    all_window_features,
    label_col="Label",
    subject_col="SubjectID",
    method_col="WindowMethod",
    target_method="event_guided",
    top_k=15,
    corr_threshold=0.90,
    max_missing_rate=0.30,
)

print("Classification summary: event-guided window-level subject-wise")
display(result_event_window.summary)

## 6. Export results

In [ ]:
# ============================================================
# 6. Export classification results
# ============================================================

result_event_subject.summary.to_csv("classification_event_subject_summary.csv", index=False)
result_event_subject.predictions.to_csv("classification_event_subject_predictions.csv", index=False)
result_event_subject.selected_feature_frequency.to_csv("classification_event_subject_selected_features.csv", index=False)

result_fixed_subject.summary.to_csv("classification_fixed_subject_summary.csv", index=False)
result_fixed_subject.predictions.to_csv("classification_fixed_subject_predictions.csv", index=False)
result_fixed_subject.selected_feature_frequency.to_csv("classification_fixed_subject_selected_features.csv", index=False)

print("Classification results exported.")